# Universal Context Engine (Converged Edition)

**Copyright 2026, Denis Rothman**

**Goal:** This notebook demonstrates the **Sovereignty of the Context Engine**.

It proves that the exact same reasoning architecture (`engine.py`) can seamlessly switch between:
1.  **Pinecone (Cloud Vector DB):** For Marketing & Legal use cases.
2.  **Oracle 23ai (Converged Enterprise DB):** For HR & Recruitment use cases.

The Engine does not change. Only the **Agents** in the Execution Plan change.

# 1. Installation & Setup

In [1]:
#@title Retrieving GitHub Private token (this cell will be deleted when the repository is public)
import os
from openai import OpenAI
from google.colab import userdata

# Load the API key from Colab secrets, set the env var, then init the client
try:
    pt = userdata.get("GITHUB_TOKEN")
    pt=pt.strip()
    if not pt:
        raise userdata.SecretNotFoundError("GITHUB_TOKEN not found.")

    print("GITHUB_TOKENAPI private token loaded successfully.")

except userdata.SecretNotFoundError:
    print('Secret "GITHUB_TOKEN" not found.')
    print('Please activate or add your GITHUB_TOKEN private token to the Colab Secrets Manager.')
except Exception as e:
    print(f"An error occurred while loading the GITHUB_TOKEN: {e}")

GITHUB_TOKENAPI private token loaded successfully.


In [2]:
# Install BOTH Pinecone and Oracle dependencies
!pip install oracledb==3.4.1 openai==2.14.0 pinecone-client==5.0.1 tiktoken==0.7.0 tqdm==4.67.1 tenacity --quiet
print("✅ Universal Dependencies installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.8/244.8 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.4/85.4 kB 6.1 MB/s eta 0:00:00
✅ Universal Dependencies installed.


In [4]:
# 2. Initialize Clients

# --- Part A: OpenAI (Mandatory) ---
try:
    # Explicitly check for the mandatory OpenAI key
    openai_key = userdata.get("API_KEY")
    if not openai_key:
        raise ValueError("OpenAI API_KEY not found in Secrets.")

    os.environ["OPENAI_API_KEY"] = openai_key
    client = OpenAI()
    print("✅ OpenAI initialized (Mandatory).")

except Exception as e:
    print(f"❌ Critical Error: OpenAI initialization failed. {e}")
    # We raise the error here because OpenAI is required for the Planner to work at all
    raise e

# --- Part B: Pinecone (Optional) ---
# Initialize variables to None so they exist even if the block fails
pc = None
pinecone_index = None

try:
    pc_key = userdata.get("PINECONE_API_KEY")
    if pc_key:
        pc = Pinecone(api_key=pc_key)
        # We only try to connect to the Index if we have a client
        pinecone_index = pc.Index("genai-mas-mcp-ch3")
        print("✅ Pinecone initialized (Optional).")
    else:
        print("ℹ️ Pinecone skipped: 'PINECONE_API_KEY' not found in Secrets.")

except Exception as e:
    # Log the warning but allow the code to continue
    print(f"⚠️ Pinecone Init Warning: {e} (Continuing without Vector DB...)")
    pc = None
    pinecone_index = None

✅ OpenAI initialized (Mandatory).
⚠️ Pinecone Init Warning: Notebook does not have access to secret PINECONE_API_KEY (Continuing without Vector DB...)


In [ ]:
#@title Retrieving engine library
import time

!curl -L -H "Authorization: token {pt}" -H "Accept: application/vnd.github.v3.raw" "https://api.github.com/repos/Denis2054/RAG-Driven-Generative-AI-2nd-Edition/contents/commons/engine/engine.zip" -o engine.zip

time.sleep(10)
!unzip -o engine.zip -d /content

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 15580  100 15580    0     0  51304      0 --:--:-- --:--:-- --:--:-- 51419
Archive:  engine.zip
  inflating: /content/agents.py      
  inflating: /content/engine.py      
  inflating: /content/helpers.py     
  inflating: /content/oracle_lib.py  
  inflating: /content/registry.py    
  inflating: /content/agent_archivist.py  
  inflating: /content/agent_recruiter.py  


In [ ]:
# 3. Import Local Modules (Uploaded Files)
try:
    from engine import context_engine
    from registry import AGENT_TOOLKIT
    from oracle_lib import OracleManager
    print("✅ Universal Engine Modules imported.")
except ImportError as e:
    print(f"❌ Module Import Failed: {e}\nUpload engine.py, registry.py, agents.py, helpers.py, oracle_lib.py, agent_archivist.py, agent_recruiter.py")

✅ Universal Engine Modules imported.


In [ ]:
# Initialize Oracle Connection (Enterprise Layer)
try:
    OracleManager.initialize()
    print("✅ Oracle 23ai Connection established.")
except Exception as e:
    print(f"⚠️ Oracle Connection Failed: {e}\n(Ignore if you only want to run Pinecone scenarios)")

✅ Oracle 23ai Connection established.


# 2. Initialize the Universal Engine

In [ ]:
# 2. Initialize the Universal Engine Wrapper
from engine import context_engine

# We define a wrapper function that pre-configures the engine with our clients and models.
# This allows us to just pass the 'goal' in the scenario steps below.
def run_universal_engine(goal):
    return context_engine(
        goal=goal,
        client=client,
        pc=pc, # Pass the Pinecone Client object, not the index
        index_name="genai-mas-mcp-ch3", # The specific index name
        generation_model="gpt-5.1",
        embedding_model="text-embedding-3-small",
        # [FIX] Updated to match the namespaces visible in Cell 13
        namespace_context="ContextLibrary",
        namespace_knowledge="KnowledgeStore"
    )

print("🚀 Universal Context Engine Wrapper is READY.")
print(AGENT_TOOLKIT.list_agents())

🚀 Universal Context Engine Wrapper is READY.

AVAILABLE CAPABILITIES:
-----------------------
1. Librarian
   - Role: Retrieves style/structure blueprints (templates).
   - Input: {"intent_query": "string description of the desired content style"}

2. Researcher
   - Role: Searches for factual information in the Pinecone Knowledge Base.
   - Input: {"topic_query": "string topic to research"}

3. Writer
   - Role: Generates final content by applying a Blueprint to Facts.
   - Input: {"blueprint": "string context", "facts": "string evidence", "previous_content": "optional string"}

4. Summarizer
   - Role: Reduces large text blocks to concise summaries.
   - Input: {"text_to_summarize": "string", "summary_objective": "string goal"}

5. OracleArchivist
   - Role: Retrieves unstructured documents (logs, reports) from the Oracle Database.
   - Input: {"intent_query": "string search query"}

6. OracleRecruiter
   - Role: Retrieves job candidates from Oracle Database using Hybrid Search (Stru

# 3. Scenario A: Marketing (Powered by Pinecone)
This execution plan uses the standard `Librarian` and `Researcher` agents connected to Pinecone.

In [ ]:
#@title Pinecone data verification
# -------------------------------------------------------------------------
print("\nIngestion complete. Final Pinecone Index Stats (may take a moment to update):")
print(pinecone_index.describe_index_stats())


Ingestion complete. Final Pinecone Index Stats (may take a moment to update):
{'dimension': 1536,
 'index_fullness': 0.0,
 'namespaces': {'ContextLibrary': {'vector_count': 3},
                'KnowledgeStore': {'vector_count': 10}},
 'total_vector_count': 13}


In [ ]:
#@title Query powered by Pinecone
pinecone_goal = "Summarize the key points of the QuantumDrive"

# We use the wrapper function we defined in the previous cell.
# The 'context_engine' function inside it will automatically call the 'planner'
# to generate the steps (Librarian -> Researcher -> Writer).
print(f"Goal: {pinecone_goal}")

# Execute the engine
result, trace = run_universal_engine(pinecone_goal)

print("\n--- FINAL MARKETING OUTPUT ---")
print(result)

Goal: Summarize the key points of the QuantumDrive

--- FINAL MARKETING OUTPUT ---
{'summary': '- **What QuantumDrive is**\n  - (Not specified in the provided text. The text is an incident log about a security breach and system outage, not a product description of QuantumDrive.)\n\n- **Core technology**\n  - Not described. No references to storage, quantum tech, distributed systems, or any “QuantumDrive” architecture—only Oracle DB logs and security systems (AuthService, DB_Connector, IDS_System).\n\n- **Main features**\n  - Not described. The text covers:\n    - Authentication events\n    - Database latency and errors\n    - Intrusion detection alerts\n    - Connection pool exhaustion\n    - Service restart by watchdog\n    - Customer tickets about downtime and slowness\n    - A phishing campaign and SQL injection attack\n  - None of these are presented as features of a product named QuantumDrive.\n\n- **Performance characteristics**\n  - Only incident-related performance issues are m

# 4. Scenario B: Recruitment (Powered by Oracle)
This execution plan switches to the `OracleRecruiter` agent. The Engine doesn't change code, only the configuration.

In [ ]:
#@title Oracle data verification
from oracle_lib import OracleManager

print("\n=== Oracle Hybrid Table Summary ===\n")

# Use the Manager's context manager instead of a raw 'connection' variable
with OracleManager.get_cursor() as cursor:
    # --- 1. Check HR Tables ---
    tables = ['CANDIDATE_POOL', 'RECRUITMENT_RULES']

    for t in tables:
        try:
            cursor.execute(f"SELECT COUNT(*) FROM {t}")
            result = cursor.fetchone()
            if result:
                print(f"Table {t}: {result[0]} rows")
        except Exception as e:
            print(f"Table {t}: Not Found or Error ({e})")

    print("\n--- Sample Candidate (Structured + Vector) ---")
    try:
        cursor.execute("""
            SELECT candidate_id, full_name, salary_expectation,
                   CASE WHEN resume_vector IS NOT NULL THEN '✅ Vector Present' ELSE '❌ No Vector' END
            FROM candidate_pool
            FETCH FIRST 2 ROWS ONLY
        """)
        for row in cursor.fetchall():
            print(row)
    except Exception as e:
        print(f"Error fetching candidates: {e}")

print("\n=== Verification Complete ===")


=== Oracle Hybrid Table Summary ===

Table CANDIDATE_POOL: 5 rows
Table RECRUITMENT_RULES: 3 rows

--- Sample Candidate (Structured + Vector) ---
('CAND_003', 'Casey M.', 210000, '✅ Vector Present')
('CAND_001', 'Alex V.', 165000, '✅ Vector Present')

=== Verification Complete ===


In [ ]:
#@title Query powered by Oracle
# 4. Scenario B: Recruitment (Powered by Oracle)
# This execution plan switches to the `OracleRecruiter` agent.
# The Engine doesn't change code, only the configuration.

# We state the constraints clearly in the goal so the Planner passes them to the OracleRecruiter.
recruitment_goal = "Find Experienced Python Developers with max salary 140k and min 3 years experience, then draft a polite rejection email for them mentioning we will keep their resume."

print(f"Goal: {recruitment_goal}")

# Execute the engine (The Planner will automatically select OracleRecruiter -> Writer)
result, trace = run_universal_engine(recruitment_goal)

print("\n--- FINAL RECRUITMENT EMAILS ---")
print(result)

Goal: Find Experienced Python Developers with max salary 140k and min 3 years experience, then draft a polite rejection email for them mentioning we will keep their resume.

--- FINAL RECRUITMENT EMAILS ---
Hey [Name],

Thanks a lot for taking the time to apply for the Experienced Python Developer role and for your interest in joining our team.

After reviewing your profile, we’ve decided to move forward with other candidates whose experience is a closer match to what we need right now. That said, we were impressed with your background, and we’ll keep your resume on file for future opportunities that might be a better fit.

Really appreciate the time and effort you put into your application, and we wish you all the best in your job search and future projects.

Best regards,  
[Your Name]  
[Your Title]  
[Company Name]


In [ ]:
# 4. Scenario B: Recruitment (Powered by Oracle)
# This execution plan switches to the `OracleRecruiter` agent.
# The Engine doesn't change code, only the configuration.

# We state the constraints clearly in the goal so the Planner passes them to the OracleRecruiter.
# UPGRADE: Explicitly used "140000" (integer format) to ensure Planner parsing accuracy.
recruitment_goal = "Find Experienced Python Developers with max salary 140000 and min 3 years experience, then draft a polite rejection email for them mentioning we will keep their resume."

print(f"Goal: {recruitment_goal}")

# Execute the engine (The Planner will automatically select OracleRecruiter -> Writer)
result, trace = run_universal_engine(recruitment_goal)

# --- RESULT HANDLING & DEBUGGING ---
if result:
    print("\n--- FINAL RECRUITMENT EMAILS ---")
    print(result)
else:
    print("\n⚠️ No Final Result returned by Engine. Inspecting Trace...")
    print(f"Trace Status: {trace.status}")

    # 1. Did we get a plan?
    if trace.plan:
        print(f"Plan Generated: {len(trace.plan)} steps")
        for step in trace.plan:
            print(f" - Step {step.get('step')}: {step.get('agent')} -> {step.get('input')}")
    else:
        print("❌ CRITICAL: No Plan was generated. Check your LLM / Planner logic.")

    # 2. Did we execute steps?
    if trace.steps:
        print(f"\nSteps Executed: {len(trace.steps)}")
        last_step = trace.steps[-1]
        print(f"Last Agent: {last_step['agent']}")
        print(f"Last Output Snippet: {str(last_step['output'])[:200]}...")
    else:
        print("❌ CRITICAL: Plan generated but no steps executed.")

Goal: Find Experienced Python Developers with max salary 140000 and min 3 years experience, then draft a polite rejection email for them mentioning we will keep their resume.

--- FINAL RECRUITMENT EMAILS ---
Subject: Application Update – Python Developer Position

Dear {{CandidateName}},

Thank you for your interest in the Python Developer position and for the time you invested in the application and interview process.

After careful consideration, we have decided to move forward with another candidate for this role. This was a difficult decision given the strength of your experience and background.

We will keep your resume on file and may reach out if we identify future opportunities that align with your skills and experience.

We appreciate your interest in our team and wish you all the best in your continued career as a Python developer.

Sincerely,  
{{YourName}}  
{{YourTitle}}  
{{CompanyName}}


In [ ]:
OracleManager.close()